In [1]:
import sys
from pprint import pprint
import os
from dotenv import load_dotenv

from ckan_batch.client import CKANClient
from ckan_batch.reader.pidinst import read_pidinst_template

In [2]:
# temp_path = r"C:\Users\mot032\CSIRO\AuScope AVRE - Persistent ID for Instruments (PIDINST)\templates\PIDINST_Batch_Instruments_Template_SingleSheet.xlsx"
# temp_path = r"C:\Users\mot032\CSIRO\AuScope AVRE - Persistent ID for Instruments (PIDINST)\data\PIDINST_Batch_BenKay.xlsx"
temp_path = r"C:\Users\mot032\CSIRO\AuScope AVRE - Persistent ID for Instruments (PIDINST)\data\PIDINST_Batch_UoM_Petrophysics_Lab.xlsx"
# temp_path = r"C:\Drive_D\projects\avre\apps\PIDINST\AuScope-instrument-registry\ckan_batch\src\ckan_batch\reader\templates\PIDINST.xlsx"

load_dotenv()

CKAN_BASE_URL = os.getenv("CKAN_BASE_URL", "").rstrip("/")
CKAN_API_KEY = os.getenv("CKAN_API_KEY")  # required for Create/Update/Delete (CUD) actions

# INITIALISE CLIENT

In [3]:
ckan_client = CKANClient(CKAN_BASE_URL, apikey=CKAN_API_KEY)

# Template Reader

In [5]:
res_map = read_pidinst_template(
    temp_path, 
    client=ckan_client,
    sheet_name="Instruments"
)
pkg_dict = res_map.records
if len(res_map.errors) > 0:
    print("=============ERRORS==========\n")
    pprint(res_map.errors)

print(len(pkg_dict))
# pprint(pkg_dict[0])

=============ERRORS==========

['[Record 2 | Row 8] Party not found: C-Therm',
 '[Record 2] Missing at least one manufacturer (MANUFACTURER.Name).']
2


In [5]:
# pprint(pkg_dict[0])

# Get all packages

In [6]:
# ckan_client.get_all(verbose=True)[0]

# Create Instruments

**NOTE:** For PIDINST set `record_type` always to "instrument", even for PLATFORMS. Both are based on the same schema, only a boolean `is_platform` separates them

In [7]:
res = ckan_client.create_records(
    records=pkg_dict,
    make_public=False,
    dry_run = False,
    record_type="instrument"
)

In [8]:
res.failed

[]

# Delete

In [7]:
pkgs = ckan_client.get_records_by_title(
    "2024 Eyre Peninsula Magnetotelluric Survey Array"
)
for pkg in pkgs:
    print(pkg.get("title"))

2024 Eyre Peninsula Magnetotelluric Survey Array


In [8]:
for pkg in pkgs:
    pkg_id = pkg.get("id")
    ckan_client.delete_record_by_id(record_id=pkg_id, hard_delete=True)

In [7]:
to_delete = ckan_client.delete_all_in_org(
    owner_org="auscope-org",
    dry_run=False,
    record_type='instrument',
    include_draft = True,
    include_private = True,
    include_public = False,
    hard_delete = True
)

Found 1 instrument record(s) in owner_org='auscope-org' for HARD DELETE (draft=True, private=True, public=False)
 - example_instrument-cgs-t-sn-000099 (46038fec-db18-4ab9-92a6-d6272de48698) [active, private] | Example Instrument

Deleted: 1


# Parties

In [10]:
# ckan_client.get_all_parties(verbose=True)

In [25]:
ckan_client.delete_all_parties(
    dry_run=False,
    hard_delete=True
)

Found 1 party group(s) for HARD DELETE
 - curtin-hub-for-immersive-visualisation-and-eresearch (8f8a16a2-70df-4cdc-91ce-f8192f320930) | Curtin Hub for Immersive Visualisation and eResearch | state=active

Deleted: 1


[{'id': '8f8a16a2-70df-4cdc-91ce-f8192f320930',
  'name': 'curtin-hub-for-immersive-visualisation-and-eresearch',
  'title': 'Curtin Hub for Immersive Visualisation and eResearch',
  'type': 'party',
  'state': 'active'}]

In [11]:
# ckan_client.action.taxonomy_list()

In [12]:
# ckan_client.action.taxonomy_term_list(id='9d93b7ac-53e5-432b-a607-408f83902525')

In [13]:
# ckan_client.action.group_list(type="facility")

In [5]:
# from ckan_batch.helpers import get_excel_template, get_notebooks

In [10]:
pkgs = ckan_client.get_all(q="test 111",verbose=True)
for pkg in pkgs:
    print(pkg.get('title'))

Test 111


In [6]:
ckan_client.get_parties_by_name()

{'adelaide university': {'name': 'adelaide-university',
  'title': 'Adelaide University',
  'roles': {'funder', 'owner'},
  'party_identifier_type': 'ROR',
  'party_identifier': 'https://ror.org/028g18b61',
  'party_contact': '',
  'alias': ''},
 'auscope': {'name': 'auscope',
  'title': 'AuScope',
  'roles': {'funder', 'owner'},
  'party_identifier_type': 'ROR',
  'party_identifier': 'https://ror.org/04s1m4564',
  'party_contact': '',
  'alias': ''},
 'curtin hub for immersive visualisation and eresearch': {'name': 'curtin-hub-for-immersive-visualisation-and-eresearch',
  'title': 'Curtin Hub for Immersive Visualisation and eResearch',
  'roles': {'funder', 'owner'},
  'party_identifier_type': 'ROR',
  'party_identifier': 'https://ror.org/00hjft591',
  'party_contact': '',
  'alias': ''},
 'curtin university': {'name': 'curtin-university',
  'title': 'Curtin University',
  'roles': {'funder', 'owner'},
  'party_identifier_type': 'ROR',
  'party_identifier': 'https://ror.org/02n415q13'

In [7]:
res = ckan_client.get_api(path='api/instrument_parties')

In [8]:
res['nodes']

[{'contact': '',
  'count': 4,
  'id': 'adelaide-university',
  'parent_id': None,
  'title': 'Adelaide University'},
 {'contact': '',
  'count': 1,
  'id': 'auscope',
  'parent_id': None,
  'title': 'AuScope'},
 {'contact': '',
  'count': 2,
  'id': 'curtin-hub-for-immersive-visualisation-and-eresearch',
  'parent_id': 'curtin-university',
  'title': 'Curtin Hub for Immersive Visualisation and eResearch'},
 {'contact': '',
  'count': 1,
  'id': 'curtin-university',
  'parent_id': None,
  'title': 'Curtin University'},
 {'contact': 'contact@update.me',
  'count': 0,
  'id': 'phoenix-geophysics',
  'parent_id': None,
  'title': 'Phoenix Geophysics'},
 {'contact': 'contact@update.me',
  'count': 0,
  'id': 'the-university-of-melbourne',
  'parent_id': None,
  'title': 'The University of Melbourne'}]

In [7]:
ckan_client.get_all(q='manufacturer_name_search:"Phoenix Geophysics (Canada)"',verbose=False)

[{'id': '6d1129df-161e-4bd3-8bb1-c2e17b3ceefd',
  'name': 'test_update_inst_party-cgs-fyi-alt_1',
  'title': 'test update inst party',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '0fbaca21-5bda-4533-9c4a-bae48055ef5a',
  'name': 'test_11-mtu-5c-1',
  'title': 'Test 11',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '1a02cb87-1099-480f-b350-2719e68f84cf',
  'name': 'test_111-cgs-fyi-10240',
  'title': 'Test 111',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '39430f76-9649-40e0-973c-d5e1908aad6a',
  'name': 'example_instrument-cgs-t-sn-111111',
  'title': 'Example Instrument',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': 'c81c6ccd-ef63-45bb-850d-4a5b110a29bd',
  'name': 'phoenix_geophysics_mtu-5c_magnetotelluric_data_acquisition_system-

In [7]:
# ckan_client.get_all(q="owner_name_search:The University of Melbourne",verbose=False)
ckan_client.get_all(q="manufacturer_name_search:Phoenix Geophysics (canada)",verbose=False)

[]

In [5]:
ckan_client.get_all(q='manufacturer_name_search:"Phoenix Geophysics (Canada)"',verbose=False)

[{'id': '39430f76-9649-40e0-973c-d5e1908aad6a',
  'name': 'example_instrument-cgs-t-sn-111111',
  'title': 'Example Instrument',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '6d1129df-161e-4bd3-8bb1-c2e17b3ceefd',
  'name': 'test_update_inst_party-cgs-fyi-alt_1',
  'title': 'test update inst party',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '0fbaca21-5bda-4533-9c4a-bae48055ef5a',
  'name': 'test_11-mtu-5c-1',
  'title': 'Test 11',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '1a02cb87-1099-480f-b350-2719e68f84cf',
  'name': 'test_111-cgs-fyi-10240',
  'title': 'Test 111',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': 'c81c6ccd-ef63-45bb-850d-4a5b110a29bd',
  'name': 'phoenix_geophysics_mtu-5c_magnetotelluric_data_acquisition_system-

In [9]:
ckan_client.get_all(q='owner_name_search:"The University of Melbourne"',verbose=False)

[{'id': 'a76d9194-fb7d-4f65-807a-38e9e5109e57',
  'name': 'tenney_junior_environmental_test_chamber-tjr-1101000034',
  'title': 'Tenney Junior Environmental Test Chamber',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '7069e5e5-e85d-41b7-9d6a-afd31b116cf9',
  'name': 'c-therm_trident_thermal_conductivity_instrument-trident-5475109',
  'title': 'C-Therm Trident Thermal Conductivity Instrument',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '21f12570-74d8-4b3f-89d1-b7e1a4f4940c',
  'name': 'abess_instruments_24_cube_style_vacuum_chamber-24alcube-50767',
  'title': 'Abess Instruments 24" Cube Style Vacuum Chamber',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'}]

In [10]:
ckan_client.get_all(q='owner_name_search_alias:"Melbourne University"',verbose=False)

[{'id': 'a76d9194-fb7d-4f65-807a-38e9e5109e57',
  'name': 'tenney_junior_environmental_test_chamber-tjr-1101000034',
  'title': 'Tenney Junior Environmental Test Chamber',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '7069e5e5-e85d-41b7-9d6a-afd31b116cf9',
  'name': 'c-therm_trident_thermal_conductivity_instrument-trident-5475109',
  'title': 'C-Therm Trident Thermal Conductivity Instrument',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'},
 {'id': '21f12570-74d8-4b3f-89d1-b7e1a4f4940c',
  'name': 'abess_instruments_24_cube_style_vacuum_chamber-24alcube-50767',
  'title': 'Abess Instruments 24" Cube Style Vacuum Chamber',
  'type': 'instrument',
  'state': 'active',
  'owner_org': '12345678-1234-1234-1234-123456789012'}]